## 0. Install packages (hard-pinned)



In [ ]:
import sys, subprocess

PY_PINS = [
    "numpy==1.26.4",
    "pandas==2.2.3",
    "scipy==1.13.1",
    "statsmodels==0.14.4",
    "scikit-learn==1.5.2",
    "scikit-bio==0.6.2",
    "biopython==1.84",
    "networkx==3.3",
    "matplotlib==3.9.2",
    "matplotlib-venn==1.1.2",
    "openpyxl==3.1.5",
    "dendropy==5.0.1",      
]

def pip_install(pins):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins], check=False)

pip_install(PY_PINS)
print("Python deps installed.")



## 1. Configuration and data loading



In [ ]:
import warnings
warnings.filterwarnings('ignore', message='Sample size too small for normal approximation')
import re, numpy as np, pandas as pd
from pathlib import Path

CANDIDATES = ["data/pairs_only_P_H.xlsx", "../data/pairs_only_P_H.xlsx",
              "pairs_only_P_H.xlsx", "pairs_only_P_H.csv"]
INPUT = next((p for p in CANDIDATES if Path(p).exists()), CANDIDATES[0])
OUTDIR = Path("sa_outputs"); OUTDIR.mkdir(exist_ok=True)
print("Reading:", INPUT)

def load_pairs(path):
    path = str(path)
    df = pd.read_excel(path, sheet_name=0) if path.lower().endswith((".xlsx",".xlsm")) else pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df.loc[:, ~df.columns.duplicated(keep="first")].copy()

df = load_pairs(INPUT)
PH_PAT = re.compile(r"^([PH])(\d+)([ab])$")
for c in df.columns:
    if PH_PAT.match(c):
        df[c] = pd.to_numeric(df[c], errors="coerce")

TAXON_COL = next((c for c in ["OTU_ID","taxon","Taxon","Lineage"] if c in df.columns), df.columns[0])
print("Shape:", df.shape, "| taxon col:", TAXON_COL)
df.head(3)

## 2. Core STRICT-ADAPTIVE criterion



In [ ]:
# ---- fixed thresholds ----
ALPHA=0.05; REL_THR=0.25; CLIFFS_THR=0.147
PREV_MIN=0.50; MIN_BASE_MED=1e-3; REL_DENOM_EPS=1e-3; WINSOR_Q=0.01; C_IQR=0.5
MIN_PAIRS_FEMALE=3; MIN_PAIRS_MALE=5; MIN_PLACEBO_PER_SEX=3

P_FEMALE_IDS={10,36,43,44,8}; P_MALE_IDS={11,16,21,31,39,45,47,48,49,51}
H_FEMALE_IDS={12,22,23,27}; H_MALE_IDS={3,7,9,14,15,16,20,21,25,29}

from scipy.stats import wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests

def collect_pairs(cols, letter):
    f={}
    for c in cols:
        m=PH_PAT.match(str(c))
        if not m: continue
        g,num,ab=m.groups()
        if g!=letter: continue
        f.setdefault(int(num),{})[ab]=c
    return {pid:(v["a"],v["b"]) for pid,v in f.items() if "a" in v and "b" in v}

def delta_matrix(df, pairs):
    a=[ac for ac,bc in pairs.values()]; b=[bc for ac,bc in pairs.values()]
    A=df[a].to_numpy(float); B=df[b].to_numpy(float)
    return B-A, A

def winsor_rows(M, q=WINSOR_Q):
    if q<=0: return M
    lo=np.nanquantile(M,q,axis=1,keepdims=True); hi=np.nanquantile(M,1-q,axis=1,keepdims=True)
    return np.clip(M,lo,hi)

def cliffs(x,y):
    x=np.sort(x[~np.isnan(x)]); y=np.sort(y[~np.isnan(y)])
    nx,ny=x.size,y.size
    if nx==0 or ny==0: return 0.0
    i=j=gt=lt=0
    while i<nx and j<ny:
        if x[i]>y[j]: gt+=nx-i; j+=1
        elif x[i]<y[j]: lt+=ny-j; i+=1
        else: i+=1; j+=1
    return (gt-lt)/(nx*ny)

def wilcox_p(d):
    d=d[~np.isnan(d)]
    if d.size==0 or np.allclose(d,0): return 1.0
    try: return float(wilcoxon(d,zero_method="wilcox",alternative="two-sided",method="auto")[1])
    except Exception: return 1.0

def analyze(df, taxon_col, p_pairs, h_pairs_sex, h_pairs_all, min_pairs):
    use_h = h_pairs_sex if len(h_pairs_sex)>=MIN_PLACEBO_PER_SEX else h_pairs_all
    dP,aP=delta_matrix(df,p_pairs); dH,aH=delta_matrix(df,use_h)
    nP=np.sum(np.isfinite(dP),axis=1); nH=np.sum(np.isfinite(dH),axis=1)
    idx=np.where((nP>=min_pairs)&(nH>=MIN_PLACEBO_PER_SEX))[0]
    dPw=winsor_rows(dP); dHw=winsor_rows(dH)
    relP=np.abs(dP)/np.maximum(np.abs(aP),REL_DENOM_EPS)
    relH=np.abs(dH)/np.maximum(np.abs(aH),REL_DENOM_EPS)
    taxa=df[taxon_col].to_numpy(); rows=[]
    for i in idx:
        dp=dPw[i][np.isfinite(dPw[i])]; dh=dHw[i][np.isfinite(dHw[i])]
        try: pPH=mannwhitneyu(dp,dh,alternative="two-sided",method="auto")[1]
        except Exception: pPH=1.0
        ap=aP[i]; ah=aH[i]
        rows.append((taxa[i],dp.size,dh.size,wilcox_p(dp),wilcox_p(dh),pPH,
            np.median(dp),np.median(dh),
            np.nanpercentile(relP[i],80),np.nanpercentile(relH[i],80),
            np.std(dp,ddof=1) if dp.size>1 else np.nan, np.std(dh,ddof=1) if dh.size>1 else np.nan,
            np.percentile(dp,75)-np.percentile(dp,25), np.percentile(dh,75)-np.percentile(dh,25),
            cliffs(dp,dh),
            np.sum(np.isfinite(ap)&(ap>0))/len(p_pairs), np.sum(np.isfinite(ah)&(ah>0))/len(use_h),
            np.nanmedian(ap[np.isfinite(ap)]) if np.isfinite(ap).any() else np.nan,
            np.nanmedian(ah[np.isfinite(ah)]) if np.isfinite(ah).any() else np.nan))
    cols=["taxon","n_pairs_P","n_pairs_H","p_P","p_H","p_PH","median_delta_P","median_delta_H",
          "q80_rel_P","q80_rel_H","sd_P","sd_H","iqr_P","iqr_H","cliffs_delta_PH",
          "prev_baseline_P","prev_baseline_H","median_a_P","median_a_H"]
    res=pd.DataFrame(rows,columns=cols)
    if res.empty: return res
    for col,src in [("p_P_adj","p_P"),("p_H_adj","p_H"),("p_PH_adj","p_PH")]:
        res[col]=multipletests(res[src].fillna(1.0),alpha=ALPHA,method="fdr_bh")[1]
    res["HL_P_ok"]=(res["iqr_H"]>0)&(res["median_delta_P"].abs()<=C_IQR*res["iqr_H"])
    res["HL_H_ok"]=(res["iqr_H"]>0)&(res["median_delta_H"].abs()<=C_IQR*res["iqr_H"])
    res["resistant_joint"]=((res["p_P_adj"]>=ALPHA)&(res["p_H_adj"]>=ALPHA)&(res["p_PH_adj"]>=ALPHA)&
        res["HL_P_ok"]&res["HL_H_ok"]&(res["q80_rel_P"]<=REL_THR)&(res["q80_rel_H"]<=REL_THR)&
        (res["cliffs_delta_PH"].abs()<CLIFFS_THR)&(res["sd_P"]<=res["sd_H"])&(res["iqr_P"]<=res["iqr_H"])&
        (res["prev_baseline_P"]>=PREV_MIN)&(res["prev_baseline_H"]>=PREV_MIN)&
        (res["median_a_P"]>=MIN_BASE_MED)&(res["median_a_H"]>=MIN_BASE_MED))
    res["HL_P_self"]=(res["iqr_P"]>0)&(res["median_delta_P"].abs()<=C_IQR*res["iqr_P"])
    res["resistant_FMF_only"]=((res["p_P_adj"]>=ALPHA)&res["HL_P_self"]&(res["q80_rel_P"]<=REL_THR)&
        (res["prev_baseline_P"]>=PREV_MIN)&(res["median_a_P"]>=MIN_BASE_MED))
    res["HL_H_self"]=(res["iqr_H"]>0)&(res["median_delta_H"].abs()<=C_IQR*res["iqr_H"])
    res["resistant_H_only"]=((res["p_H_adj"]>=ALPHA)&res["HL_H_self"]&(res["q80_rel_H"]<=REL_THR)&
        (res["prev_baseline_H"]>=PREV_MIN)&(res["median_a_H"]>=MIN_BASE_MED))
    return res

def run_all(df):
    pall=collect_pairs(df.columns,"P"); hall=collect_pairs(df.columns,"H")
    pf={i:pall[i] for i in sorted(P_FEMALE_IDS&set(pall))}
    pm={i:pall[i] for i in sorted(P_MALE_IDS&set(pall))}
    hf={i:hall[i] for i in hall if i in H_FEMALE_IDS}
    hm={i:hall[i] for i in hall if i in H_MALE_IDS}
    print(f"P pairs: total={len(pall)} | F={len(pf)} M={len(pm)}")
    print(f"H pairs: total={len(hall)} | F={len(hf)} M={len(hm)}")
    return analyze(df,TAXON_COL,pf,hf,hall,MIN_PAIRS_FEMALE), analyze(df,TAXON_COL,pm,hm,hall,MIN_PAIRS_MALE)

female, male = run_all(df)
TOTAL = len(df)
print(f"\nFemale: joint={int(female.resistant_joint.sum())} ({100*female.resistant_joint.mean():.1f}% of tested)")
print(f"Male:   joint={int(male.resistant_joint.sum())} ({100*male.resistant_joint.mean():.1f}% of tested)")
female.to_csv(OUTDIR/"female_metrics.csv",index=False)
male.to_csv(OUTDIR/"male_metrics.csv",index=False)

## 3. Duplication diagnosis — joint vs cohort-specific



In [ ]:
def diagnose(res, label):
    j = set(res.loc[res.resistant_joint, "taxon"])
    f = set(res.loc[res.resistant_FMF_only, "taxon"])
    h = set(res.loc[res.resistant_H_only, "taxon"])
    jac = len(f & h) / max(len(f | h), 1)
    print(f"[{label}] tested={len(res)}")
    print(f"  joint (manuscript)      : {len(j)}")
    print(f"  FMF-only (diagnostic)   : {len(f)}")
    print(f"  Healthy-only (diagnostic): {len(h)}")
    print(f"  FMF-only ∩ Healthy-only : {len(f & h)}  (Jaccard={jac:.3f})")
    print(f"  joint ⊆ (FMF-only ∩ H-only)? {j <= (f & h)}")
    print()
    return dict(label=label, tested=len(res), joint=len(j),
                fmf_only=len(f), h_only=len(h), shared=len(f & h), jaccard=round(jac,3))

import pandas as pd
diag = pd.DataFrame([diagnose(female,"FEMALE"), diagnose(male,"MALE")])
diag.to_csv(OUTDIR/"duplication_diagnosis.csv", index=False)
diag

## 4. CLR / compositionally-aware robustness check (corrected)



In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr

def clr_transform_columns(df):
    out = df.copy()
    cols = [c for c in df.columns if PH_PAT.match(str(c))]
    M = out[cols].to_numpy(float)
    M = M - np.nanmin(M) + 1.0
    logM = np.log(M)
    gm = np.nanmean(logM, axis=0, keepdims=True)
    out[cols] = logM - gm
    return out

def resistant_invariant(res):
    return ((res["p_P_adj"]>=ALPHA)&(res["p_H_adj"]>=ALPHA)&(res["p_PH_adj"]>=ALPHA)&
            res["HL_P_ok"]&res["HL_H_ok"]&
            (res["cliffs_delta_PH"].abs()<CLIFFS_THR)&
            (res["sd_P"]<=res["sd_H"])&(res["iqr_P"]<=res["iqr_H"]))

df_clr = clr_transform_columns(df)
import io as _io, contextlib
with contextlib.redirect_stdout(_io.StringIO()):
    female_clr, male_clr = run_all(df_clr)

rows=[]
for name, raw, clr in [("FEMALE", female, female_clr), ("MALE", male, male_clr)]:
    a = set(raw.loc[resistant_invariant(raw), "taxon"])
    b = set(clr.loc[resistant_invariant(clr), "taxon"])
    jac = len(a & b) / max(len(a | b), 1)
    rec = len(a & b) / max(len(a), 1)
    merged = raw[["taxon","cliffs_delta_PH"]].merge(
        clr[["taxon","cliffs_delta_PH"]], on="taxon", suffixes=("_raw","_clr"))
    rho, p = spearmanr(merged["cliffs_delta_PH_raw"], merged["cliffs_delta_PH_clr"])
    print(f"[{name}] invariant-set: raw={len(a)} clr={len(b)} shared={len(a&b)} "
          f"recall={rec:.2f} Jaccard={jac:.2f} | Spearman rho(Cliff's delta raw vs CLR)={rho:.3f} (p={p:.1e})")
    rows.append(dict(sex=name, raw_set=len(a), clr_set=len(b), shared=len(a&b),
                     recall=round(rec,3), jaccard=round(jac,3),
                     spearman_rho=round(rho,3)))

clr_tab = pd.DataFrame(rows)
clr_tab.to_csv(OUTDIR/"clr_sensitivity.csv", index=False)

clr_tab

## 5. Threshold justification (Cliff's δ and C_IQR sweep)



In [ ]:
def sweep_cliffs(res, grid):
    base=((res["p_P_adj"]>=ALPHA)&(res["p_H_adj"]>=ALPHA)&(res["p_PH_adj"]>=ALPHA)&
          res["HL_P_ok"]&res["HL_H_ok"]&(res["q80_rel_P"]<=REL_THR)&(res["q80_rel_H"]<=REL_THR)&
          (res["sd_P"]<=res["sd_H"])&(res["iqr_P"]<=res["iqr_H"])&
          (res["prev_baseline_P"]>=PREV_MIN)&(res["prev_baseline_H"]>=PREV_MIN)&
          (res["median_a_P"]>=MIN_BASE_MED)&(res["median_a_H"]>=MIN_BASE_MED))
    return [int((base&(res["cliffs_delta_PH"].abs()<t)).sum()) for t in grid]

def sweep_ciqr(res, grid):
    out=[]
    for k in grid:
        hlP=(res["iqr_H"]>0)&(res["median_delta_P"].abs()<=k*res["iqr_H"])
        hlH=(res["iqr_H"]>0)&(res["median_delta_H"].abs()<=k*res["iqr_H"])
        f=((res["p_P_adj"]>=ALPHA)&(res["p_H_adj"]>=ALPHA)&(res["p_PH_adj"]>=ALPHA)&hlP&hlH&
           (res["q80_rel_P"]<=REL_THR)&(res["q80_rel_H"]<=REL_THR)&
           (res["cliffs_delta_PH"].abs()<CLIFFS_THR)&(res["sd_P"]<=res["sd_H"])&(res["iqr_P"]<=res["iqr_H"])&
           (res["prev_baseline_P"]>=PREV_MIN)&(res["prev_baseline_H"]>=PREV_MIN)&
           (res["median_a_P"]>=MIN_BASE_MED)&(res["median_a_H"]>=MIN_BASE_MED))
        out.append(int(f.sum()))
    return out

import numpy as np, matplotlib.pyplot as plt
cliff_grid=np.round(np.arange(0.05,0.36,0.02),3)
ciqr_grid =np.round(np.arange(0.25,1.05,0.05),3)
fig,ax=plt.subplots(1,2,figsize=(12,4),dpi=150)
for res,name in [(female,"Female"),(male,"Male")]:
    ax[0].plot(cliff_grid,sweep_cliffs(res,cliff_grid),marker="o",label=name)
    ax[1].plot(ciqr_grid, sweep_ciqr(res, ciqr_grid), marker="o",label=name)
ax[0].axvline(0.147,ls="--",c="grey"); ax[0].set_title("Cliff's δ threshold"); ax[0].set_xlabel("|δ| cutoff"); ax[0].set_ylabel("resistant taxa"); ax[0].legend()
ax[1].axvline(0.5,ls="--",c="grey");   ax[1].set_title("C_IQR threshold");   ax[1].set_xlabel("C_IQR"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUTDIR/"threshold_sweep.png",bbox_inches="tight"); plt.show()

## 6. LOSO + subject bootstrap internal validation



In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore", message="Sample size too small for normal approximation")

def run_all_silent(df_in):
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        return run_all(df_in)

def resistant_set(df_in, sex):
    f,m = run_all_silent(df_in)
    res = f if sex=="F" else m
    return set(res.loc[res.resistant_joint,"taxon"])

def drop_subject_columns(df_in, letter, pid):
    cols=[c for c in df_in.columns if PH_PAT.match(str(c))
          and PH_PAT.match(str(c)).group(1)==letter
          and int(PH_PAT.match(str(c)).group(2))==pid]
    return df_in.drop(columns=cols)

def loso_frequencies(df_in, sex):
    full = resistant_set(df_in, sex)
    pset = sorted(P_FEMALE_IDS if sex=="F" else P_MALE_IDS)
    hset = sorted(H_FEMALE_IDS if sex=="F" else H_MALE_IDS)
    drops = [("P",p) for p in pset] + [("H",h) for h in hset]
    recs=[]; freq={}
    for k,(letter,pid) in enumerate(drops,1):
        s = resistant_set(drop_subject_columns(df_in,letter,pid), sex)
        inter=len(full & s)
        recs.append(dict(dropped=f"{letter}{pid}", n=len(s),
                         jaccard=len(full&s)/max(len(full|s),1),
                         recall=inter/max(len(full),1),
                         precision=inter/max(len(s),1)))
        for t in s: freq[t]=freq.get(t,0)+1
        print(f"  [{sex}] fold {k}/{len(drops)} dropped {letter}{pid}: |set|={len(s)}")
    nfold=len(drops)
    freq={t:c/nfold for t,c in freq.items()}
    return full, pd.DataFrame(recs), freq

loso_results={}
for sex,name in [("F","FEMALE"),("M","MALE")]:
    print(f"Running LOSO for {name} ...")
    full, rec, freq = loso_frequencies(df, sex)
    rec.to_csv(OUTDIR/f"loso_{name.lower()}.csv", index=False)
    loso_results[sex]=dict(full=full, rec=rec, freq=freq)
    print(f"[{name}] full={len(full)} | median Jaccard={rec.jaccard.median():.3f} "
          f"(IQR {rec.jaccard.quantile(.25):.3f}-{rec.jaccard.quantile(.75):.3f})\n")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CORE_THRESHOLD = 0.7   
grid = np.round(np.arange(0.5, 0.96, 0.05), 2)

fig, ax = plt.subplots(figsize=(8,5), dpi=150)
core_sizes={}
rows=[]
for sex,name,color in [("F","Female","#c0392b"),("M","Male","#2c7fb8")]:
    freq=loso_results[sex]["freq"]; full=loso_results[sex]["full"]
    sizes=[sum(1 for v in freq.values() if v>=t) for t in grid]
    ax.plot(grid, sizes, marker="o", color=color, label=f"{name} (full={len(full)})")
    core={t for t,v in freq.items() if v>=CORE_THRESHOLD}
    core_sizes[sex]=core
    for t,sz in zip(grid,sizes):
        rows.append(dict(sex=name, threshold=t, core_size=sz))
    print(f"{name}: stable core @≥{CORE_THRESHOLD} = {len(core)} taxa "
          f"({100*len(core)/max(len(full),1):.0f}% of full list)")

ax.axvline(CORE_THRESHOLD, ls="--", c="grey", alpha=.7)
ax.set_xlabel("LOSO inclusion threshold"); ax.set_ylabel("stable core size (taxa)")
ax.set_title("Consensus curve: reproducible core vs inclusion threshold")
ax.legend(); plt.tight_layout()
plt.savefig(OUTDIR/"consensus_curve.png", dpi=200, bbox_inches="tight"); plt.show()
pd.DataFrame(rows).to_csv(OUTDIR/"consensus_curve.csv", index=False)

core_female = core_sizes["F"]; core_male = core_sizes["M"]
def annotate_core(core_set, res):
    sub=res[res["taxon"].isin(core_set)][["taxon"]].copy()
    ann=df[[TAXON_COL,"Genus","Family","Phylum"]].rename(columns={TAXON_COL:"taxon"})
    return sub.merge(ann,on="taxon",how="left")
annotate_core(core_female, female).to_csv(OUTDIR/"stable_core_female.csv", index=False)
annotate_core(core_male,   male  ).to_csv(OUTDIR/"stable_core_male.csv", index=False)
print(f"\nHEADLINE: stable cores — Female={len(core_female)}, Male={len(core_male)}, "
      f"shared={len(core_female & core_male)}")

## 7. Genus profile trees — quantitative metrics, bootstrap support, tip-controlled test



In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from Bio.Phylo.TreeConstruction import DistanceMatrix as BioDM, DistanceTreeConstructor
import dendropy

FEATS = ["median_delta_P","median_delta_H","q80_rel_P","q80_rel_H",
         "prev_baseline_P","prev_baseline_H","sd_P","sd_H",
         "iqr_P","iqr_H","cliffs_delta_PH"]

def genus_matrix(res, df_annot, taxon_subset=None):
    mask = res["taxon"].isin(taxon_subset) if taxon_subset is not None else res.resistant_joint
    sub = res.loc[mask, ["taxon"]+ [c for c in FEATS if c in res.columns]].copy()
    g = df_annot[[TAXON_COL,"Genus"]].rename(columns={TAXON_COL:"taxon"})
    sub = sub.merge(g, on="taxon", how="left").dropna(subset=["Genus"])
    sub = sub[~sub["Genus"].astype(str).isin(["","nan","unclassified","Unclassified"])]
    return sub.groupby("Genus")[[c for c in FEATS if c in sub.columns]].mean()

def nj_tree_newick(mat):
    if mat.shape[0] < 3: return None
    Xz = StandardScaler().fit_transform(mat.values)
    D = squareform(pdist(Xz, metric="euclidean"))
    names = [f"{i}_{str(x)}" for i,x in enumerate(mat.index)]
    lower = [list(D[i,:i+1]) for i in range(len(names))]
    tree = DistanceTreeConstructor().nj(BioDM(names, lower))
    from io import StringIO; from Bio import Phylo
    s=StringIO(); Phylo.write(tree,s,"newick"); return s.getvalue()

def tree_metrics(newick):
    t=dendropy.Tree.get(data=newick, schema="newick")
    t.encode_bipartitions()
    pdm=t.phylogenetic_distance_matrix()
    taxa=list(t.taxon_namespace)
    pw=[pdm.patristic_distance(a,b) for i,a in enumerate(taxa) for b in taxa[i+1:]]
    try: t.reroot_at_midpoint(update_bipartitions=False)
    except Exception: pass
    rt=[leaf.distance_from_root() for leaf in t.leaf_node_iter()]
    tbl=sum(e.length or 0 for e in t.edges())
    n=len(taxa)
    # nearest-tip
    nnt=[]
    for a in taxa:
        ds=[pdm.patristic_distance(a,b) for b in taxa if b is not a]
        if ds: nnt.append(min(ds))
    # Sackin / Colless
    def sackin(tree):
        return sum(len(leaf.ancestor_iter()) for leaf in tree.leaf_node_iter() for _ in [0])
    return dict(n_tips=n,
                mean_patristic=float(np.mean(pw)) if pw else np.nan,
                mean_root_to_tip=float(np.mean(rt)) if rt else np.nan,
                total_branch_length=float(tbl),
                tip_density=float(n/tbl) if tbl>0 else np.nan,
                mean_nearest_tip=float(np.mean(nnt)) if nnt else np.nan)

core_F = core_female if ('core_female' in globals() and len(core_female)>=10) else set(female.loc[female.resistant_joint,'taxon'])
core_M = core_male   if ('core_male'   in globals() and len(core_male)  >=10) else set(male.loc[male.resistant_joint,'taxon'])
print(f"Trees built on: Female n={len(core_F)}  Male n={len(core_M)}  (stable core where available)")
gm_F=genus_matrix(female, df, core_F); gm_M=genus_matrix(male, df, core_M)
nwk_F=nj_tree_newick(gm_F); nwk_M=nj_tree_newick(gm_M)
mF=tree_metrics(nwk_F); mM=tree_metrics(nwk_M)
obs=pd.DataFrame([{**{"sex":"Female"},**mF},{**{"sex":"Male"},**mM}])
obs.to_csv(OUTDIR/"tree_metrics_observed.csv",index=False)
print(obs.to_string(index=False))

In [ ]:
import numpy as np, pandas as pd
rng=np.random.default_rng(7)

def downsample_metric(mat, k, reps=200, metric="mean_patristic"):
    vals=[]
    idx=np.arange(mat.shape[0])
    for _ in range(reps):
        take=rng.choice(idx, size=k, replace=False)
        nwk=nj_tree_newick(mat.iloc[take])
        if nwk: vals.append(tree_metrics(nwk)[metric])
    return np.array(vals)

k=min(gm_F.shape[0], gm_M.shape[0])
print(f"Equalizing tip count to k={k} genera (min of {gm_F.shape[0]} F, {gm_M.shape[0]} M)")
for metric in ["mean_patristic","tip_density","total_branch_length","mean_nearest_tip"]:
    vF=downsample_metric(gm_F,k,metric=metric); vM=downsample_metric(gm_M,k,metric=metric)
    diff=np.mean(vF)-np.mean(vM)
    pooled=pd.concat([gm_F,gm_M], ignore_index=False).reset_index(drop=True); n=pooled.shape[0]
    null=[]
    for _ in range(200):
        perm=rng.permutation(n)
        a=pooled.iloc[perm[:k]]; b=pooled.iloc[perm[k:2*k]]
        na=nj_tree_newick(a); nb=nj_tree_newick(b)
        if na and nb: null.append(tree_metrics(na)[metric]-tree_metrics(nb)[metric])
    null=np.array(null)
    p=(1+np.sum(np.abs(null)>=abs(diff)))/(len(null)+1)
    print(f"  {metric:22s} F={np.mean(vF):.3f} M={np.mean(vM):.3f} diff={diff:+.3f} perm_p={p:.3f}")

## 8. Figure 1 (hi-res) and Figure 2 (Venn)



In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

setF=set(female.loc[female.resistant_joint,"taxon"]); setM=set(male.loc[male.resistant_joint,"taxon"])
cF = core_female if 'core_female' in globals() else setF
cM = core_male   if 'core_male'   in globals() else setM

fig,axes=plt.subplots(1,2,figsize=(12,5),dpi=300)
venn2(subsets=(len(setF-setM),len(setM-setF),len(setF&setM)),
      set_labels=("Female (raw)","Male (raw)"), ax=axes[0])
axes[0].set_title(f"Raw joint lists\n(female={len(setF)}, male={len(setM)})")
venn2(subsets=(len(cF-cM),len(cM-cF),len(cF&cM)),
      set_labels=("Female core","Male core"), ax=axes[1])
axes[1].set_title(f"Reproducible stable cores (LOSO≥0.7)\n(female={len(cF)}, male={len(cM)})")
fig.suptitle("Placebo-nonresponsive taxa: overlap between sexes (stable in FMF and Healthy)")
plt.tight_layout(); plt.savefig(OUTDIR/"figure2_venn.png",dpi=300,bbox_inches="tight"); plt.show()
print(f"RAW  female-only={len(setF-setM)} shared={len(setF&setM)} male-only={len(setM-setF)}")
print(f"CORE female-only={len(cF-cM)} shared={len(cF&cM)} male-only={len(cM-cF)}")

### 8b. Taxonomic composition of the stable core (replaces qualitative tree comparison)



In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

cF = core_female if 'core_female' in globals() else set(female.loc[female.resistant_joint,'taxon'])
cM = core_male   if 'core_male'   in globals() else set(male.loc[male.resistant_joint,'taxon'])

def core_composition(core_set, level="Family", top=12):
    ann = df[[TAXON_COL, level]].rename(columns={TAXON_COL:"taxon"})
    sub = ann[ann["taxon"].isin(core_set)].copy()
    sub[level] = sub[level].astype(str).replace({"nan":"unclassified","":"unclassified"})
    return sub[level].value_counts().head(top)

fig, axes = plt.subplots(1, 2, figsize=(13,6), dpi=300)
for ax,(core,name,color) in zip(axes, [(cF,"Female core","#c0392b"),(cM,"Male core","#2c7fb8")]):
    vc = core_composition(core, "Family", 12).iloc[::-1]
    ax.barh(range(len(vc)), vc.values, color=color)
    ax.set_yticks(range(len(vc))); ax.set_yticklabels(vc.index, fontsize=9)
    ax.set_xlabel("resistant taxa (count)")
    ax.set_title(f"{name}: top families (n={len(core)})")
plt.suptitle("Taxonomic composition of the placebo-nonresponsive stable core")
plt.tight_layout(); plt.savefig(OUTDIR/"figure_core_composition.png", dpi=300, bbox_inches="tight"); plt.show()

for name,core in [("female",cF),("male",cM)]:
    for level in ["Family","Genus"]:
        core_composition(core, level, 30).rename("count").to_csv(OUTDIR/f"core_{name}_{level.lower()}_top.csv")
print("Saved composition figure + top-family/genus tables for both cores.")

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def panel_data(pairs):
    a=[ac for ac,bc in pairs.values()]; b=[bc for ac,bc in pairs.values()]
    A=df[a].to_numpy(float); B=df[b].to_numpy(float)
    return np.nanmedian(A,axis=1), np.nanmedian(B,axis=1), np.nanmedian(B-A,axis=1)

pall=collect_pairs(df.columns,"P"); hall=collect_pairs(df.columns,"H")
groups=[("P♀",{i:pall[i] for i in sorted(P_FEMALE_IDS&set(pall))}),
        ("P♂",{i:pall[i] for i in sorted(P_MALE_IDS&set(pall))}),
        ("H♀",{i:hall[i] for i in H_FEMALE_IDS if i in hall}),
        ("H♂",{i:hall[i] for i in H_MALE_IDS if i in hall})]
fig,axes=plt.subplots(len(groups),2,figsize=(13,12),dpi=300)
for r,(name,pairs) in enumerate(groups):
    mA,mB,mD=panel_data(pairs); order=np.argsort(mB)[::-1]
    axes[r,0].plot(mB[order],lw=0.4,label="B (post)"); axes[r,0].plot(mA[order],lw=0.4,alpha=.6,label="A (pre)")
    axes[r,0].set_title(f"{name}  median A/B (n={len(pairs)} pairs)"); axes[r,0].legend(fontsize=7)
    axes[r,1].hist(mD[np.isfinite(mD)],bins=120); axes[r,1].set_title(f"{name}  Δ=B−A distribution")
plt.tight_layout(); plt.savefig(OUTDIR/"figure1_hires.png",dpi=300,bbox_inches="tight"); plt.show()
print("Saved figure1_hires.png (300 dpi)")

## 9. Benchmark vs compositionally-aware DA methods (Python)



In [ ]:
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.multitest import multipletests

def _clr_matrix(M):
    M = M - np.nanmin(M) + 1.0
    L = np.log(M)
    return L - np.nanmean(L, axis=0, keepdims=True)   

def aldex_like_stable(df, pairs):
    a=[ac for ac,bc in pairs.values()]; b=[bc for ac,bc in pairs.values()]
    A=_clr_matrix(df[a].to_numpy(float)); B=_clr_matrix(df[b].to_numpy(float))
    ps=[]
    for i in range(A.shape[0]):
        d=B[i]-A[i]; d=d[np.isfinite(d)]
        if d.size<3 or np.allclose(d,0): ps.append(1.0)
        else:
            try: ps.append(wilcoxon(d)[1])
            except Exception: ps.append(1.0)
    q=multipletests(np.nan_to_num(ps,nan=1.0),method="fdr_bh")[1]
    eff=np.nanmedian(B-A,axis=1)/(np.nanstd(np.concatenate([A,B],axis=1),axis=1)+1e-9)
    stable=(q>=0.05)&(np.abs(eff)<0.5)
    return set(df.loc[stable, TAXON_COL])

def ancombc_like_stable(df, pairs):
    a=[ac for ac,bc in pairs.values()]; b=[bc for ac,bc in pairs.values()]
    A=_clr_matrix(df[a].to_numpy(float)); B=_clr_matrix(df[b].to_numpy(float))
    ps=[]
    for i in range(A.shape[0]):
        x,y=A[i],B[i]; m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<3: ps.append(1.0)
        else:
            try: ps.append(ttest_rel(y[m],x[m])[1])
            except Exception: ps.append(1.0)
    q=multipletests(np.nan_to_num(ps,nan=1.0),method="fdr_bh")[1]
    return set(df.loc[q>=0.05, TAXON_COL])

pall=collect_pairs(df.columns,"P"); hall=collect_pairs(df.columns,"H")
groups={
    "fmf_female":    ({i:pall[i] for i in sorted(P_FEMALE_IDS&set(pall))}, "female"),
    "fmf_male":      ({i:pall[i] for i in sorted(P_MALE_IDS&set(pall))},   "male"),
    "healthy_female":({i:hall[i] for i in H_FEMALE_IDS if i in hall},      "female"),
    "healthy_male":  ({i:hall[i] for i in H_MALE_IDS if i in hall},        "male"),
}
sa={"female":set(female.loc[female.resistant_joint,"taxon"]),
    "male":  set(male.loc[male.resistant_joint,"taxon"])}
core={"female": core_female if "core_female" in globals() else sa["female"],
      "male":   core_male   if "core_male"   in globals() else sa["male"]}

rows=[]
for tag,(pairs,sex) in groups.items():
    ald=aldex_like_stable(df,pairs); bc=ancombc_like_stable(df,pairs)
    for tool,st in [("ALDEx2-like",ald),("ANCOMBC-like",bc)]:
        for ref_name,ref in [("SA_joint",sa[sex]),("SA_core",core[sex])]:
            inter=len(ref&st)
            rows.append(dict(group=tag,tool=tool,reference=ref_name,
                ref_n=len(ref),tool_stable=len(st),shared=inter,
                recall_of_ref=round(inter/max(len(ref),1),3),
                jaccard=round(inter/max(len(ref|st),1),3)))
bench=pd.DataFrame(rows)
bench.to_csv(OUTDIR/"benchmark_overlap_python.csv",index=False)
print(bench.to_string(index=False))
bench

## 10. Summary tables for the manuscript / response letter



In [ ]:
import pandas as pd
summary = {
 "total_taxa": int(TOTAL),
 "female_resistant_joint": int(female.resistant_joint.sum()),
 "male_resistant_joint": int(male.resistant_joint.sum()),
 "female_pct_of_total": round(100*female.resistant_joint.sum()/TOTAL,2),
 "male_pct_of_total": round(100*male.resistant_joint.sum()/TOTAL,2),
 "shared_between_sexes": len(set(female.loc[female.resistant_joint,"taxon"]) &
                             set(male.loc[male.resistant_joint,"taxon"])),
 "stable_core_female": len(core_female) if 'core_female' in globals() else None,
 "stable_core_male": len(core_male) if 'core_male' in globals() else None,
 "loso_median_jaccard_female": round(loso_results['F']['rec'].jaccard.median(),3) if 'loso_results' in globals() else None,
 "loso_median_jaccard_male": round(loso_results['M']['rec'].jaccard.median(),3) if 'loso_results' in globals() else None,
}
print("=== Headline numbers ===")
for k,v in summary.items(): print(f"  {k}: {v}")
pd.Series(summary).to_csv(OUTDIR/"headline_summary.csv")
